[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S20_ml_modelos_fuertes.ipynb)

# Sesión 20 · Modelos más fuertes

**Módulo 5: Machine Learning** · ⏱️ Duración estimada: 60 a 90 minutos (el módulo 5 prevé 1 o 2 días por sesión)

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Reconocer el sobreajuste comparando el desempeño en entrenamiento y en prueba.
2. Buscar hiperparámetros con `GridSearchCV` usando validación cruzada.
3. Entrenar un Random Forest y un modelo de gradient boosting con LightGBM.
4. Comparar modelos de forma justa y leer las importancias de las variables.

## 📋 Qué debes saber antes
Sesiones 17 a 19: `fit` y `predict`, árboles de decisión, AUC y validación cruzada.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Algunas celdas tardan unos segundos: están entrenando decenas de modelos.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas. LightGBM ya viene instalado en Colab.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica: ¿el cliente pagará su préstamo? ----------
_n = 3000
_ing = np.round(rng.lognormal(1.3, 0.5, _n), 2)
_di = np.round(rng.beta(2, 5, _n) * 1.2, 3)
_edad = rng.integers(20, 71, _n)
_emp = rng.integers(0, 241, _n)
_atr = rng.poisson(0.4, _n)
_uso = np.round(rng.uniform(0, 1, _n), 3)
_ncred = rng.integers(0, 7, _n)
_logit = (-2.8 + 2.2 * ((_di > 0.45) & (_atr > 0)) + 1.2 * (_di > 0.6) + 0.5 * _atr + 1.4 * (_uso > 0.85)
          + 0.0015 * (_edad - 45) ** 2 - 0.004 * _emp - 0.15 * _ing)
creditos = pd.DataFrame({
    "ingreso_miles": _ing, "deuda_ingreso": _di, "edad": _edad, "meses_empleo": _emp,
    "atrasos_12m": _atr, "uso_tarjeta": _uso, "n_creditos": _ncred,
    "incumple": (rng.random(_n) < 1 / (1 + np.exp(-_logit))).astype(int),
})
VARIABLES = ["ingreso_miles", "deuda_ingreso", "edad", "meses_empleo", "atrasos_12m", "uso_tarjeta", "n_creditos"]
PROFUNDIDADES = [1, 2, 3, 4, 6, 8, 12, 20]

_D = copy.deepcopy({"creditos": creditos})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _auc(real, prob):
    """AUC con la fórmula de rangos de Mann-Whitney (sin scikit-learn); los empates reciben el rango promedio."""
    real, prob = [int(v) for v in real], [float(v) for v in prob]
    orden = sorted(range(len(prob)), key=lambda i: prob[i])
    rangos = [0.0] * len(prob)
    i = 0
    while i < len(orden):
        j = i
        while j + 1 < len(orden) and prob[orden[j + 1]] == prob[orden[i]]:
            j += 1
        for k in range(i, j + 1):
            rangos[orden[k]] = (i + j) / 2 + 1
        i = j + 1
    pos = [rangos[i] for i in range(len(real)) if real[i] == 1]
    n_pos, n_neg = len(pos), len(real) - len(pos)
    return (math.fsum(pos) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def _particion():
    """El X_train, X_test, y_train, y_test del alumno, si existen y son coherentes."""
    v = [globals().get(n) for n in ("X_train", "X_test", "y_train", "y_test")]
    if all(isinstance(a, pd.DataFrame) for a in v[:2]) and all(isinstance(a, pd.Series) for a in v[2:]) and len(v[0]) == len(v[2]) and len(v[1]) == len(v[3]):
        return v
    return None


def _auc_modelo(m, X, y):
    return _auc(y.tolist(), m.predict_proba(X)[:, 1])


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    p = _particion()
    if p is None:
        r.mal("Faltan `X_train`, `X_test`, `y_train` o `y_test`.")
        r.fin()
        return
    xs, xt, ys, yt = p
    if len(xt) != 750 or [str(c) for c in xs.columns] != VARIABLES or abs(ys.mean() - yt.mean()) > 0.005:
        r.mal("La partición debería dejar 25 % para prueba, estratificada, con las columnas de `VARIABLES`.")
    else:
        r.ok("La partición es correcta.")
    from sklearn.tree import DecisionTreeClassifier
    filas = []
    for d in PROFUNDIDADES:
        t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(xs, ys)
        filas.append([_auc_modelo(t, xs, ys), _auc_modelo(t, xt, yt)])
    _df(r, "curva", ["auc_train", "auc_test"], filas,
        "un árbol por profundidad (`random_state=42`), con su AUC en entrenamiento y en prueba", indice=PROFUNDIDADES, tol=1e-9)
    pruebas = [f[1] for f in filas]
    _esc(r, "mejor_profundidad", PROFUNDIDADES[pruebas.index(max(pruebas))], "la profundidad con mayor AUC en prueba")
    _esc(r, "brecha_20", filas[-1][0] - filas[-1][1], "AUC de entrenamiento menos AUC de prueba con profundidad 20", tol=1e-9)
    ax = _grafico(r, "ax_curva")
    if ax is not None:
        lineas = [l for l in ax.get_lines() if len(l.get_ydata()) == len(PROFUNDIDADES)]
        leyenda = ax.get_legend()
        if len(lineas) != 2 or not any(_cerca_lista(l.get_ydata(), [f[0] for f in filas], 1e-9) for l in lineas) \
                or not any(_cerca_lista(l.get_ydata(), pruebas, 1e-9) for l in lineas):
            r.mal("`ax_curva` debería tener dos líneas: el AUC de entrenamiento y el de prueba según la profundidad.")
        elif leyenda is None or len(leyenda.get_texts()) != 2:
            r.mal("Agrega una leyenda para las dos líneas.")
        else:
            r.ok("`ax_curva` muestra dónde empieza el sobreajuste.")
        _rotulos(r, "ax_curva", ax, None, "Profundidad máxima", "AUC")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_train_20": "b399e5afcd7822a4082968fd88be832ae2163fda3da62ed3f6d9402bfba8dd1c",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    p = _particion()
    b = r.var("busqueda")
    if p is None:
        r.mal("Primero resuelve el ejercicio 1.")
    elif b is not _FALTA:
        xs, xt, ys, yt = p
        grilla = {"max_depth": [2, 3, 4, 5, 6, 8], "min_samples_leaf": [1, 5, 20, 50]}
        if type(b).__name__ != "GridSearchCV" or not hasattr(b, "best_params_"):
            r.mal("`busqueda` debería ser un `GridSearchCV` ya entrenado.")
        elif b.param_grid != grilla:
            r.mal("La grilla de `busqueda` no es la pedida: revisa los valores de `max_depth` y `min_samples_leaf`.")
        elif b.scoring != "roc_auc" or getattr(b.cv, "n_splits", None) != 5:
            r.mal("`busqueda` debería usar `scoring=\"roc_auc\"` y el `StratifiedKFold` de 5 pliegues.")
        else:
            res = b.cv_results_
            mejor = int(np.argmax(res["mean_test_score"]))
            r.ok("`busqueda` probó la grilla pedida con validación cruzada.")
            r.valor("mejores_param", res["params"][mejor], dict, "debería ser `busqueda.best_params_`")
            _esc(r, "mejor_auc_cv", float(res["mean_test_score"][mejor]), "el AUC promedio de validación cruzada de la mejor combinación", tol=1e-9)
            _esc(r, "auc_test_arbol", _auc_modelo(b.best_estimator_, xt, yt), "el AUC en prueba del mejor árbol (usa `busqueda.predict_proba`)", tol=1e-9)
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_n_entrenamientos": "9e796e90ece60516d85344278c5c5193a804ec32a23bd370133285c79a580f4b",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    p = _particion()
    m = r.var("rf")
    if p is None:
        r.mal("Primero resuelve el ejercicio 1.")
    elif m is not _FALTA:
        xs, xt, ys, yt = p
        if type(m).__name__ != "RandomForestClassifier" or not hasattr(m, "estimators_"):
            r.mal("`rf` debería ser un `RandomForestClassifier` entrenado.")
        elif (m.n_estimators, m.min_samples_leaf, m.random_state) != (300, 5, 42):
            r.mal("`rf` debería tener `n_estimators=300`, `min_samples_leaf=5` y `random_state=42`.")
        else:
            r.ok("`rf` es un bosque de 300 árboles.")
            _esc(r, "auc_rf", _auc_modelo(m, xt, yt), "el AUC del bosque en prueba", tol=1e-9)
            imp = r.var("importancias_rf")
            if imp is not _FALTA:
                ref = dict(zip(VARIABLES, m.feature_importances_))
                if not isinstance(imp, pd.Series) or sorted(map(str, imp.index)) != sorted(VARIABLES) \
                        or not all(abs(float(imp[k]) - ref[k]) < 1e-12 for k in VARIABLES):
                    r.mal("`importancias_rf` debería tener `rf.feature_importances_` con los nombres de las variables como índice.")
                elif any(a < b for a, b in zip(imp.tolist(), imp.tolist()[1:])):
                    r.mal("`importancias_rf` debería estar ordenada de mayor a menor.")
                else:
                    r.ok("`importancias_rf` es correcta.")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_mismos_datos": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    p = _particion()
    if p is None:
        r.mal("Primero resuelve el ejercicio 1.")
        r.fin()
        return
    xs, xt, ys, yt = p
    aucs = {}
    for nombre, tipo, var_auc, clave in (("lgbm", "LGBMClassifier", "auc_lgbm", "lightgbm"), ("modelo_log", "LogisticRegression", "auc_log", "logística")):
        m = r.var(nombre)
        if m is _FALTA:
            continue
        if type(m).__name__ != tipo or not hasattr(m, "classes_"):
            r.mal(f"`{nombre}` debería ser un `{tipo}` entrenado.")
            continue
        if nombre == "lgbm" and (m.n_estimators, m.learning_rate, m.num_leaves) != (300, 0.05, 15):
            r.mal("`lgbm` debería tener `n_estimators=300`, `learning_rate=0.05` y `num_leaves=15`.")
            continue
        aucs[clave] = _auc_modelo(m, xt, yt)
        _esc(r, var_auc, aucs[clave], f"el AUC de `{nombre}` en prueba", tol=1e-9)
    b, rf = globals().get("busqueda"), globals().get("rf")
    if hasattr(b, "best_estimator_"):
        aucs["árbol (grid)"] = _auc_modelo(b.best_estimator_, xt, yt)
    if hasattr(rf, "estimators_"):
        aucs["random forest"] = _auc_modelo(rf, xt, yt)
    c = r.var("comparacion")
    if c is not _FALTA and len(aucs) == 4:
        orden = sorted(aucs, key=lambda k: -aucs[k])
        _ser(r, "comparacion", [aucs[k] for k in orden], "el AUC en prueba de los cuatro modelos, de mayor a menor", indice=orden, tol=1e-9)
    elif c is not _FALTA:
        r.mal("Para revisar `comparacion` necesito los cuatro modelos bien entrenados: corrige primero los puntos marcados.")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_boosting": "0a0dd55e0012c0fb4415a1af7f7afab16852efad1511ca488030614fb1430742",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    p = _particion()
    b = r.var("busqueda_lgbm")
    if p is None:
        r.mal("Primero resuelve el ejercicio 1.")
    elif b is not _FALTA:
        xs, xt, ys, yt = p
        grilla = {"learning_rate": [0.03, 0.1], "num_leaves": [7, 15, 31], "n_estimators": [100, 300]}
        if type(b).__name__ != "GridSearchCV" or not hasattr(b, "best_params_") or type(b.estimator).__name__ != "LGBMClassifier":
            r.mal("`busqueda_lgbm` debería ser un `GridSearchCV` entrenado sobre un `LGBMClassifier`.")
        elif b.param_grid != grilla or b.scoring != "roc_auc" or getattr(b.cv, "n_splits", None) != 3:
            r.mal("Revisa la grilla, `scoring=\"roc_auc\"` y los 3 pliegues estratificados de `busqueda_lgbm`.")
        else:
            res = b.cv_results_
            mejor = int(np.argmax(res["mean_test_score"]))
            r.ok("`busqueda_lgbm` probó la grilla pedida.")
            r.valor("mejores_param_lgbm", res["params"][mejor], dict, "debería ser `busqueda_lgbm.best_params_`")
            _esc(r, "auc_cv_lgbm", float(res["mean_test_score"][mejor]), "el mejor AUC de validación cruzada", tol=1e-9)
            auc = _auc_modelo(b.best_estimator_, xt, yt)
            _esc(r, "auc_lgbm_ajustado", auc, "el AUC en prueba del mejor LightGBM", tol=1e-9)
            m = globals().get("modelo_log")
            if hasattr(m, "classes_"):
                _esc(r, "mejora_vs_log", auc - _auc_modelo(m, xt, yt), "el AUC del LightGBM ajustado menos el de la logística", tol=1e-9)
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    rf = globals().get("rf")
    ax = _grafico(r, "ax_imp")
    if ax is not None and hasattr(rf, "feature_importances_"):
        imp = sorted(zip(VARIABLES, rf.feature_importances_), key=lambda kv: kv[1])
        barras = _barras(ax)
        colores = [_hex(b.get_facecolor()) for b in barras]
        if len(barras) != len(VARIABLES) or not _cerca_lista([b.get_width() for b in barras], [v for _, v in imp], 1e-12):
            r.mal("`ax_imp` debería tener barras horizontales con las importancias del bosque, la mayor arriba.")
        elif colores != [GRIS] * (len(imp) - 1) + [AZUL]:
            r.mal("Destaca la variable más importante en `AZUL` y deja el resto en `GRIS`.")
        else:
            r.ok("`ax_imp` muestra las importancias con la principal destacada.")
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
`creditos`: 3000 préstamos con el ingreso del cliente (en miles de soles), su deuda como proporción del ingreso, edad, meses en su empleo, atrasos del último año, uso de su línea de tarjeta (0 a 1), cantidad de créditos vigentes y si **incumplió** el pago (1) o no (0). `VARIABLES` tiene las siete variables y `PROFUNDIDADES`, las profundidades que vas a probar.

El riesgo real depende de **combinaciones** de variables (por ejemplo, mucha deuda **y** atrasos a la vez): justo lo que los árboles capturan mejor que una regresión logística.

In [ ]:
print(creditos.head(), "\n")
print(creditos["incumple"].mean().round(3), "\n")
print(creditos.groupby("incumple")[VARIABLES].mean().round(2))

---
## 1. Sobreajuste: memorizar no es aprender

### 📘 Concepto
Un modelo **sobreajusta** (*overfitting*) cuando aprende tan bien los detalles del entrenamiento, incluido el ruido, que empeora con datos nuevos. La señal es una **brecha**: le va muy bien en entrenamiento y bastante peor en prueba.

Un árbol sin límite de profundidad puede crear una hoja para casi cada cliente: su AUC de entrenamiento se acerca a 1, pero en prueba cae. Con muy poca profundidad pasa lo contrario: el modelo es demasiado simple y rinde poco en los dos conjuntos (*subajuste*). Entre ambos extremos hay un punto intermedio que generaliza mejor.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

gen_ej = np.random.default_rng(1)
X_ej = pd.DataFrame({"x": gen_ej.normal(size=300)})
y_ej = pd.Series((X_ej["x"] + gen_ej.normal(scale=1.5, size=300) > 0).astype(int))
for d in [1, 3, 15]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_ej[:200], y_ej[:200])
    print(d, round(roc_auc_score(y_ej[:200], t.predict_proba(X_ej[:200])[:, 1]), 3),
          round(roc_auc_score(y_ej[200:], t.predict_proba(X_ej[200:])[:, 1]), 3))

### ✍️ Tu turno · Ejercicio 1: la curva del sobreajuste
**Parte A.**
1. `X` (columnas de `VARIABLES`) e `y` (`incumple`), y `X_train`, `X_test`, `y_train`, `y_test` con 25 % para prueba, `random_state=42` y estratificado.
2. `curva`: un DataFrame con una fila por profundidad de `PROFUNDIDADES` (como índice) y las columnas `auc_train` y `auc_test`: el AUC de un árbol con esa `max_depth` (y `random_state=42`) en entrenamiento y en prueba.
3. `mejor_profundidad`: la profundidad con mayor AUC en prueba, y `brecha_20`: el AUC de entrenamiento menos el de prueba con profundidad 20.
4. `fig_curva, ax_curva`: las dos columnas de `curva` como líneas según la profundidad, con leyenda (`entrenamiento` y `prueba`), eje x `Profundidad máxima` y eje y `AUC`.

**Parte B.** Predice **sin ejecutar**: `pred_train_20` = el AUC aproximado en entrenamiento del árbol de profundidad 20 (un número).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Recorre `PROFUNDIDADES` con un `for`: en cada vuelta entrena un árbol y guarda sus dos AUC en una lista.
</details>

<details><summary>💡 Pista 2</summary>

`curva = pd.DataFrame(filas, index=PROFUNDIDADES, columns=["auc_train", "auc_test"])`. La mejor profundidad es `curva["auc_test"].idxmax()`.
</details>

---
## 2. Hiperparámetros y `GridSearchCV`

### 📘 Concepto
Los **hiperparámetros** son las perillas que se fijan **antes** de entrenar (`max_depth`, `min_samples_leaf`...). Elegirlos mirando el conjunto de prueba es hacer trampa: la prueba dejaría de ser una evaluación imparcial. La forma honesta es la **validación cruzada dentro del entrenamiento**.

`GridSearchCV` prueba todas las combinaciones de una grilla, evalúa cada una con validación cruzada y reentrena la mejor con todo el entrenamiento:

```python
from sklearn.model_selection import GridSearchCV, StratifiedKFold
grilla = {"max_depth": [2, 3], "min_samples_leaf": [1, 20]}
busqueda = GridSearchCV(DecisionTreeClassifier(random_state=42), grilla, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc")
busqueda.fit(X_train, y_train)
busqueda.best_params_, busqueda.best_score_      # mejor combinación y su AUC de validación cruzada
busqueda.predict_proba(X_test)                   # usa el mejor modelo, ya reentrenado
```

`min_samples_leaf` exige un mínimo de casos en cada hoja: evita reglas basadas en uno o dos clientes.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

busqueda_ej = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": [1, 3, 15]},
                           cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").fit(X_ej, y_ej)
print(busqueda_ej.best_params_, round(busqueda_ej.best_score_, 3))
print(pd.DataFrame(busqueda_ej.cv_results_)[["params", "mean_test_score"]])

### ✍️ Tu turno · Ejercicio 2: buscar el mejor árbol
**Parte A.**
1. `busqueda`: un `GridSearchCV` sobre `DecisionTreeClassifier(random_state=42)` con la grilla `{"max_depth": [2, 3, 4, 5, 6, 8], "min_samples_leaf": [1, 5, 20, 50]}`, validación cruzada `StratifiedKFold(5, shuffle=True, random_state=42)` y `scoring="roc_auc"`, entrenado con los datos de entrenamiento.
2. `mejores_param` y `mejor_auc_cv`: la mejor combinación y su AUC de validación cruzada.
3. `auc_test_arbol`: el AUC en prueba del mejor árbol.

¿Se parece `mejor_auc_cv` a `auc_test_arbol`? ¿Mejoró respecto de la mejor profundidad de `curva`?

**Parte B.** Predice **sin ejecutar**: `pred_n_entrenamientos` = cuántos árboles entrena la búsqueda durante la validación cruzada (sin contar el reentrenamiento final).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Copia la estructura del concepto con la grilla pedida. `busqueda.predict_proba(X_test)` ya usa el mejor modelo.
</details>

<details><summary>💡 Pista 2</summary>

Para la parte B: combinaciones de la grilla por cantidad de pliegues.
</details>

---
## 3. Random Forest: muchos árboles votan

### 📘 Concepto
Un árbol solo es inestable: un pequeño cambio en los datos cambia mucho sus reglas. Un **Random Forest** entrena cientos de árboles distintos y promedia sus probabilidades:
- cada árbol aprende con una **muestra al azar** de los clientes (con reemplazo);
- en cada división, cada árbol mira solo un **subconjunto al azar** de las variables.

Como los árboles se equivocan en cosas distintas, al promediar los errores se compensan. Suele funcionar bien sin mucho ajuste. `n_jobs=-1` usa todos los procesadores para entrenar más rápido, y `feature_importances_` promedia la importancia de cada variable en todos los árboles.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_ej = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X_ej[:200], y_ej[:200])
print(round(roc_auc_score(y_ej[200:], rf_ej.predict_proba(X_ej[200:])[:, 1]), 3), len(rf_ej.estimators_))

### ✍️ Tu turno · Ejercicio 3: un bosque
**Parte A.**
1. `rf`: un `RandomForestClassifier` con `n_estimators=300`, `min_samples_leaf=5`, `random_state=42` y `n_jobs=-1`, entrenado.
2. `auc_rf`: su AUC en prueba.
3. `importancias_rf`: una Series con las importancias y los nombres de las variables como índice, de mayor a menor.

¿Qué variables usa más el bosque? ¿Tiene sentido para el negocio?

**Parte B.** Responde en `pred_mismos_datos` con `"sí"` o `"no"`: ¿todos los árboles del bosque aprenden con exactamente los mismos clientes?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Es la misma API: crear, `fit` y `predict_proba`.
</details>

<details><summary>💡 Pista 2</summary>

`pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)`.
</details>

---
## 4. Gradient boosting con LightGBM

### 📘 Concepto
El **boosting** también combina muchos árboles, pero en **secuencia**: cada árbol nuevo, pequeño, se concentra en corregir los errores de los anteriores. Es la familia de modelos que suele ganar en datos tabulares. **LightGBM** es una implementación rápida con la misma API de scikit-learn:

```python
from lightgbm import LGBMClassifier
lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, random_state=42, verbose=-1)
```

- `n_estimators`: cuántos árboles se suman.
- `learning_rate`: cuánto corrige cada árbol. Más bajo es más prudente, pero necesita más árboles.
- `num_leaves`: la complejidad de cada árbol.
- `verbose=-1` oculta los mensajes de entrenamiento.

Muchos árboles con un `learning_rate` alto también sobreajustan: por eso sus hiperparámetros se eligen con validación cruzada.

In [ ]:
from lightgbm import LGBMClassifier

lgbm_ej = LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=7, random_state=0, verbose=-1).fit(X_ej[:200], y_ej[:200])
print(round(roc_auc_score(y_ej[200:], lgbm_ej.predict_proba(X_ej[200:])[:, 1]), 3))

### ✍️ Tu turno · Ejercicio 4: el boosting y la tabla de posiciones
**Parte A.**
1. `lgbm`: un `LGBMClassifier` con `n_estimators=300`, `learning_rate=0.05`, `num_leaves=15`, `random_state=42` y `verbose=-1`, entrenado; `auc_lgbm`: su AUC en prueba.
2. `modelo_log`: una regresión logística (`max_iter=1000`) como referencia; `auc_log`: su AUC en prueba.
3. `comparacion`: una Series con el AUC en prueba de `"logística"`, `"árbol (grid)"`, `"random forest"` y `"lightgbm"`, ordenada de mayor a menor.

**Parte B.** Responde en `pred_boosting` con `"paralelo"` o `"secuencia"`: ¿cómo se entrenan los árboles del boosting?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Arma un diccionario con los cuatro AUC y conviértelo en Series.
</details>

<details><summary>💡 Pista 2</summary>

`pd.Series({"logística": auc_log, "árbol (grid)": auc_test_arbol, "random forest": auc_rf, "lightgbm": auc_lgbm}).sort_values(ascending=False)`.
</details>

---
## 🏋️ Reto final: ajustar LightGBM
1. `busqueda_lgbm`: un `GridSearchCV` sobre `LGBMClassifier(random_state=42, verbose=-1)` con la grilla `{"learning_rate": [0.03, 0.1], "num_leaves": [7, 15, 31], "n_estimators": [100, 300]}`, validación cruzada `StratifiedKFold(3, shuffle=True, random_state=42)` y `scoring="roc_auc"`, entrenado.
2. `mejores_param_lgbm` y `auc_cv_lgbm`: la mejor combinación y su AUC de validación cruzada.
3. `auc_lgbm_ajustado`: el AUC en prueba del mejor modelo.
4. `mejora_vs_log`: cuánto le gana en AUC a la regresión logística.

Escribe en una celda de texto por qué elegiste los hiperparámetros con validación cruzada y no mirando `auc_lgbm_ajustado`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Es el ejercicio 2 con otro modelo y otra grilla.
</details>

<details><summary>💡 Pista 2</summary>

`busqueda_lgbm.predict_proba(X_test)[:, 1]` da las probabilidades del mejor modelo reentrenado.
</details>

---
## 🚀 Nivel pro (opcional): graficar las importancias
Crea `fig_imp, ax_imp` con barras horizontales de las importancias de `rf`, la mayor arriba, con la variable más importante en `AZUL` y el resto en `GRIS`, y un título que diga cuál es. Recuerda: la importancia dice cuánto usa el modelo cada variable, no si la relación es causal.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Reconocer el sobreajuste y el subajuste comparando entrenamiento y prueba.
- [ ] Explicar qué es un hiperparámetro y por qué no se elige mirando la prueba.
- [ ] Usar `GridSearchCV` y leer `best_params_`, `best_score_` y `cv_results_`.
- [ ] Explicar en qué se diferencian un Random Forest y el boosting.
- [ ] Entrenar LightGBM y explicar `n_estimators`, `learning_rate` y `num_leaves`.
- [ ] Comparar varios modelos en una tabla y explicar qué dicen (y qué no) las importancias.

**Próxima sesión (S21):** feature engineering: codificar categorías, escalar, crear variables, evitar el data leakage y armar `Pipeline`.